# Task 3: News Sentiment and Stock Movement Correlation

This notebook scores headline sentiment, aligns news with trading days, computes returns, and measures Pearson correlation.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf

from src.correlation import pearson_correlation, sentiment_return_dataset
from src.data_loading import load_news, load_price_data
from src.sentiment import add_sentiment_scores, classify_sentiment
from src.visualization import plot_sentiment_return_scatter, save_current_figure

sns.set_theme(style='whitegrid')
RAW = Path('../data/raw')
FIGURES = Path('../reports/figures')
FIGURES.mkdir(parents=True, exist_ok=True)

## Load News and Price Data

In [ ]:
NEWS_FILE = RAW / 'fnspid_news.csv'
TICKERS = ['AAPL', 'MSFT', 'GOOGL']

news = load_news(NEWS_FILE)
news = news[news['stock'].isin(TICKERS)].copy()
news_sentiment = add_sentiment_scores(news)
news_sentiment[['headline', 'stock', 'date', 'sentiment_score', 'sentiment_label']].head()

In [ ]:
price_frames = []
for ticker in TICKERS:
    price_file = RAW / 'prices' / f'{ticker}.csv'
    if price_file.exists():
        frame = load_price_data(price_file)
    else:
        frame = yf.download(ticker, start='2020-01-01', progress=False).reset_index()
    frame['stock'] = ticker
    price_frames.append(frame)

prices = __import__('pandas').concat(price_frames, ignore_index=True)
prices.head()

## Align Sentiment With Daily Returns

In [ ]:
analysis = sentiment_return_dataset(news_sentiment, prices)
analysis['sentiment_category'] = analysis['avg_sentiment'].map(classify_sentiment)
analysis.head()

In [ ]:
corr = pearson_correlation(analysis)
corr

In [ ]:
plot_sentiment_return_scatter(analysis, corr)
save_current_figure(FIGURES / 'sentiment_return_scatter.png')

In [ ]:
category_returns = analysis.groupby('sentiment_category')['daily_return_pct'].mean().reindex(['negative', 'neutral', 'positive'])
fig, ax = plt.subplots(figsize=(7, 4))
category_returns.plot(kind='bar', ax=ax)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('Average Daily Return by Sentiment Category')
ax.set_xlabel('Sentiment category')
ax.set_ylabel('Average daily return (%)')
save_current_figure(FIGURES / 'return_by_sentiment_category.png')
category_returns

## Interpretation Draft

TextBlob was selected because it provides a lightweight, reproducible polarity score for short text without requiring model downloads. The Pearson coefficient above measures linear association between average daily headline sentiment and same-day stock return after aligning non-trading-day articles to the next trading day.

Replace this paragraph after running the notebook: interpret the sign and magnitude of the coefficient, compare average returns across negative/neutral/positive sentiment days, and discuss limitations such as lagged reactions, earnings cycles, market-wide shocks, repeated syndicated headlines, and the fact that correlation does not prove causation.